# Introduction to Large Language Models (LLMs)

This notebook covers the foundational concepts of how Large Language Models work, including:
- Text tokenization
- Word embeddings
- Transformer architecture
- Self-attention mechanism
- Positional encoding
- Training and inference basics

## What is a Large Language Model?

A Large Language Model (LLM) is a neural network trained on massive amounts of text data to understand and generate human-like text. Key characteristics:

- **Scale**: Billions to trillions of parameters
- **Architecture**: Based on Transformer architecture
- **Training**: Self-supervised learning on diverse text data
- **Capabilities**: Text generation, translation, summarization, question answering, coding, and more

### Examples of LLMs:
- GPT (Generative Pre-trained Transformer) series
- BERT (Bidirectional Encoder Representations from Transformers)
- Claude
- LLaMA
- PaLM

## 1. Tokenization

Tokenization is the process of breaking text into smaller units (tokens) that the model can process. Tokens can be words, subwords, or characters.

In [ ]:
# Simple word tokenization example
text = "Large Language Models are powerful AI systems."

# Basic whitespace tokenization
tokens = text.split()
print("Tokens:", tokens)
print("Number of tokens:", len(tokens))

### Subword Tokenization

Modern LLMs use subword tokenization (like BPE, WordPiece, or SentencePiece) to handle:
- Rare words
- Out-of-vocabulary words
- Different languages
- Efficient vocabulary size

In [ ]:
# Simulating subword tokenization
text = "unhappiness"

# A real tokenizer might split this as:
subword_tokens = ["un", "happiness"]
print(f"Original word: {text}")
print(f"Subword tokens: {subword_tokens}")

# This allows the model to understand:
# - 'un-' prefix (negation)
# - 'happiness' (known word)
# Even if 'unhappiness' wasn't in training data

In [ ]:
# Building a simple vocabulary
corpus = [
    "I love machine learning",
    "Machine learning is powerful",
    "I love AI and deep learning"
]

# Create vocabulary from corpus
vocabulary = set()
for sentence in corpus:
    vocabulary.update(sentence.lower().split())

# Convert to sorted list and add special tokens
vocabulary = ['<PAD>', '<UNK>', '<START>', '<END>'] + sorted(list(vocabulary))

# Create token-to-id mapping
token_to_id = {token: idx for idx, token in enumerate(vocabulary)}
id_to_token = {idx: token for token, idx in token_to_id.items()}

print("Vocabulary size:", len(vocabulary))
print("\nVocabulary:", vocabulary)
print("\nToken to ID mapping:")
for token, idx in list(token_to_id.items())[:10]:
    print(f"  {token}: {idx}")

In [ ]:
# Encoding text to token IDs
def encode(text, token_to_id):
    tokens = text.lower().split()
    return [token_to_id.get(token, token_to_id['<UNK>']) for token in tokens]

# Decoding token IDs back to text
def decode(token_ids, id_to_token):
    return ' '.join([id_to_token[idx] for idx in token_ids])

# Test encoding and decoding
test_text = "I love deep learning"
encoded = encode(test_text, token_to_id)
decoded = decode(encoded, id_to_token)

print(f"Original: {test_text}")
print(f"Encoded: {encoded}")
print(f"Decoded: {decoded}")

## 2. Word Embeddings

Word embeddings convert tokens into dense vectors of real numbers. These vectors capture semantic meaning - similar words have similar vectors.

**Why embeddings?**
- Represent words in continuous vector space
- Capture semantic relationships
- Enable mathematical operations on words
- More efficient than one-hot encoding

In [ ]:
import numpy as np

# Simple embedding example
vocab_size = len(vocabulary)
embedding_dim = 8  # Small dimension for illustration

# Initialize random embeddings (in practice, these are learned)
np.random.seed(42)
embedding_matrix = np.random.randn(vocab_size, embedding_dim) * 0.1

print(f"Embedding matrix shape: {embedding_matrix.shape}")
print(f"Vocabulary size: {vocab_size}")
print(f"Embedding dimension: {embedding_dim}")

# Get embedding for a specific word
word = "learning"
word_id = token_to_id[word]
word_embedding = embedding_matrix[word_id]

print(f"\nEmbedding for '{word}':")
print(word_embedding)

In [ ]:
# Embedding lookup function
def get_embeddings(token_ids, embedding_matrix):
    """Convert token IDs to embeddings"""
    return embedding_matrix[token_ids]

# Example: embed a sentence
sentence = "I love machine learning"
token_ids = encode(sentence, token_to_id)
sentence_embeddings = get_embeddings(token_ids, embedding_matrix)

print(f"Sentence: {sentence}")
print(f"Token IDs: {token_ids}")
print(f"\nEmbeddings shape: {sentence_embeddings.shape}")
print(f"Each token is represented by a {embedding_dim}-dimensional vector")

In [ ]:
# Computing similarity between words using cosine similarity
def cosine_similarity(vec1, vec2):
    """Calculate cosine similarity between two vectors"""
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    return dot_product / (norm1 * norm2)

# Compare similarity between words
word1 = "machine"
word2 = "learning"
word3 = "love"

emb1 = embedding_matrix[token_to_id[word1]]
emb2 = embedding_matrix[token_to_id[word2]]
emb3 = embedding_matrix[token_to_id[word3]]

sim_12 = cosine_similarity(emb1, emb2)
sim_13 = cosine_similarity(emb1, emb3)

print(f"Similarity between '{word1}' and '{word2}': {sim_12:.4f}")
print(f"Similarity between '{word1}' and '{word3}': {sim_13:.4f}")
print("\nNote: These are random embeddings. In trained models, related words have higher similarity.")

## 3. Transformer Architecture

The Transformer is the architecture behind modern LLMs. Key components:

1. **Input Embeddings**: Convert tokens to vectors
2. **Positional Encoding**: Add position information
3. **Multi-Head Self-Attention**: Learn relationships between tokens
4. **Feed-Forward Networks**: Process each position
5. **Layer Normalization**: Stabilize training
6. **Residual Connections**: Enable deep networks

### Transformer Benefits:
- Parallel processing (faster than RNNs)
- Long-range dependencies
- Scalable to billions of parameters

## 4. Positional Encoding

Since Transformers process all tokens in parallel, we need to add positional information so the model knows the order of words.

In [ ]:
def get_positional_encoding(seq_len, d_model):
    """
    Create positional encoding matrix
    
    Args:
        seq_len: Sequence length
        d_model: Embedding dimension
    
    Returns:
        Positional encoding matrix of shape (seq_len, d_model)
    """
    position = np.arange(seq_len)[:, np.newaxis]  # (seq_len, 1)
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    
    pos_encoding = np.zeros((seq_len, d_model))
    pos_encoding[:, 0::2] = np.sin(position * div_term)  # Even indices
    pos_encoding[:, 1::2] = np.cos(position * div_term)  # Odd indices
    
    return pos_encoding

# Example
seq_length = 10
d_model = 8

pos_enc = get_positional_encoding(seq_length, d_model)
print(f"Positional encoding shape: {pos_enc.shape}")
print(f"\nPositional encoding for position 0:")
print(pos_enc[0])
print(f"\nPositional encoding for position 5:")
print(pos_enc[5])

In [ ]:
# Visualizing positional encodings
import matplotlib.pyplot as plt

# Create larger positional encoding for visualization
seq_len = 50
d_model = 128
pos_enc_vis = get_positional_encoding(seq_len, d_model)

plt.figure(figsize=(12, 6))
plt.imshow(pos_enc_vis.T, cmap='RdBu', aspect='auto', interpolation='nearest')
plt.colorbar(label='Value')
plt.xlabel('Position in Sequence')
plt.ylabel('Embedding Dimension')
plt.title('Positional Encoding Pattern')
plt.tight_layout()
plt.show()

print("Each column represents the positional encoding for a position in the sequence.")
print("The pattern allows the model to understand token positions.")

## 5. Self-Attention Mechanism

Self-attention is the core innovation of Transformers. It allows each word to "attend to" (focus on) other words in the sequence.

### How it works:
1. Create Query (Q), Key (K), Value (V) vectors for each token
2. Calculate attention scores: how much each word should attend to others
3. Apply softmax to get attention weights
4. Compute weighted sum of values

**Formula**: Attention(Q, K, V) = softmax(QK^T / √d_k) V

In [ ]:
def softmax(x, axis=-1):
    """Compute softmax values"""
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Compute scaled dot-product attention
    
    Args:
        Q: Query matrix (seq_len, d_k)
        K: Key matrix (seq_len, d_k)
        V: Value matrix (seq_len, d_v)
        mask: Optional mask to prevent attention to certain positions
    
    Returns:
        output: Attention output (seq_len, d_v)
        attention_weights: Attention weights (seq_len, seq_len)
    """
    d_k = Q.shape[-1]
    
    # Calculate attention scores
    scores = np.matmul(Q, K.T) / np.sqrt(d_k)
    
    # Apply mask if provided (e.g., for causal attention)
    if mask is not None:
        scores = scores + (mask * -1e9)
    
    # Apply softmax to get attention weights
    attention_weights = softmax(scores, axis=-1)
    
    # Compute weighted sum of values
    output = np.matmul(attention_weights, V)
    
    return output, attention_weights

print("Attention function defined successfully!")

In [ ]:
# Example: Self-Attention on a simple sequence
np.random.seed(42)

# Sequence of 4 tokens, each with 8-dimensional embeddings
seq_len = 4
d_model = 8

# Simulated input embeddings
X = np.random.randn(seq_len, d_model)

# Initialize Q, K, V projection matrices (in practice, these are learned)
d_k = d_model
W_q = np.random.randn(d_model, d_k) * 0.1
W_k = np.random.randn(d_model, d_k) * 0.1
W_v = np.random.randn(d_model, d_k) * 0.1

# Compute Q, K, V
Q = np.matmul(X, W_q)
K = np.matmul(X, W_k)
V = np.matmul(X, W_v)

print(f"Input shape: {X.shape}")
print(f"Q shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")

In [ ]:
# Apply attention
output, attention_weights = scaled_dot_product_attention(Q, K, V)

print(f"\nAttention output shape: {output.shape}")
print(f"Attention weights shape: {attention_weights.shape}")
print(f"\nAttention weights (each row sums to 1):")
print(attention_weights)
print(f"\nSum of each row: {attention_weights.sum(axis=1)}")

In [ ]:
# Visualize attention weights
plt.figure(figsize=(8, 6))
plt.imshow(attention_weights, cmap='viridis', aspect='auto')
plt.colorbar(label='Attention Weight')
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.title('Attention Weights Matrix')

# Add values to cells
for i in range(seq_len):
    for j in range(seq_len):
        plt.text(j, i, f'{attention_weights[i, j]:.2f}',
                ha='center', va='center', color='white')

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Each row shows how much a query token attends to all key tokens")
print("- Brighter colors indicate higher attention weights")

### Causal (Masked) Attention

In autoregressive models (like GPT), tokens can only attend to previous tokens, not future ones. This is achieved using a causal mask.

In [ ]:
def create_causal_mask(size):
    """Create a causal mask to prevent attending to future positions"""
    mask = np.triu(np.ones((size, size)), k=1)
    return mask

# Create and visualize causal mask
causal_mask = create_causal_mask(seq_len)

print("Causal Mask (1 = masked position):")
print(causal_mask)

# Apply causal attention
output_causal, attention_weights_causal = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

print("\nCausal Attention Weights:")
print(attention_weights_causal)
print("\nNotice: Each position only attends to itself and previous positions")

In [ ]:
# Visualize causal attention
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Regular attention
im1 = ax1.imshow(attention_weights, cmap='viridis', aspect='auto')
ax1.set_title('Regular Self-Attention')
ax1.set_xlabel('Key Position')
ax1.set_ylabel('Query Position')
plt.colorbar(im1, ax=ax1)

# Causal attention
im2 = ax2.imshow(attention_weights_causal, cmap='viridis', aspect='auto')
ax2.set_title('Causal (Masked) Self-Attention')
ax2.set_xlabel('Key Position')
ax2.set_ylabel('Query Position')
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

print("Causal attention ensures each token can only look at previous tokens")
print("This is essential for language generation (predicting next token)")

## 6. Multi-Head Attention

Instead of performing a single attention operation, multi-head attention runs multiple attention operations in parallel, allowing the model to focus on different aspects of the input.

In [ ]:
def multi_head_attention(X, num_heads, d_model):
    """
    Simplified multi-head attention
    
    Args:
        X: Input tensor (seq_len, d_model)
        num_heads: Number of attention heads
        d_model: Model dimension
    
    Returns:
        output: Multi-head attention output
        all_attention_weights: List of attention weights from each head
    """
    seq_len = X.shape[0]
    d_k = d_model // num_heads
    
    # Initialize projection matrices for each head
    np.random.seed(42)
    heads_output = []
    all_attention_weights = []
    
    for head in range(num_heads):
        # Create Q, K, V for this head
        W_q = np.random.randn(d_model, d_k) * 0.1
        W_k = np.random.randn(d_model, d_k) * 0.1
        W_v = np.random.randn(d_model, d_k) * 0.1
        
        Q = np.matmul(X, W_q)
        K = np.matmul(X, W_k)
        V = np.matmul(X, W_v)
        
        # Apply attention
        head_output, attention_weights = scaled_dot_product_attention(Q, K, V)
        heads_output.append(head_output)
        all_attention_weights.append(attention_weights)
    
    # Concatenate all heads
    multi_head_output = np.concatenate(heads_output, axis=-1)
    
    # Final linear projection
    W_o = np.random.randn(d_model, d_model) * 0.1
    output = np.matmul(multi_head_output, W_o)
    
    return output, all_attention_weights

# Example
num_heads = 2
multi_output, head_attentions = multi_head_attention(X, num_heads, d_model)

print(f"Multi-head attention output shape: {multi_output.shape}")
print(f"Number of attention heads: {num_heads}")
print(f"\nAttention weights from head 1:")
print(head_attentions[0])
print(f"\nAttention weights from head 2:")
print(head_attentions[1])

## 7. Feed-Forward Network

After attention, each position passes through a feed-forward network independently.

In [ ]:
def relu(x):
    """ReLU activation function"""
    return np.maximum(0, x)

def feed_forward_network(x, d_model, d_ff):
    """
    Position-wise feed-forward network
    
    FFN(x) = ReLU(xW1 + b1)W2 + b2
    
    Args:
        x: Input (seq_len, d_model)
        d_model: Model dimension
        d_ff: Feed-forward dimension (typically 4 * d_model)
    """
    np.random.seed(42)
    
    # First linear layer
    W1 = np.random.randn(d_model, d_ff) * 0.1
    b1 = np.zeros(d_ff)
    hidden = relu(np.matmul(x, W1) + b1)
    
    # Second linear layer
    W2 = np.random.randn(d_ff, d_model) * 0.1
    b2 = np.zeros(d_model)
    output = np.matmul(hidden, W2) + b2
    
    return output

# Example
d_ff = 32  # Typically 4 * d_model
ff_output = feed_forward_network(multi_output, d_model, d_ff)

print(f"Feed-forward input shape: {multi_output.shape}")
print(f"Feed-forward output shape: {ff_output.shape}")
print(f"Hidden dimension: {d_ff}")

## 8. Layer Normalization and Residual Connections

These techniques stabilize training and enable deeper networks.

In [ ]:
def layer_norm(x, epsilon=1e-6):
    """
    Layer normalization
    
    Normalizes across the feature dimension for each sample
    """
    mean = np.mean(x, axis=-1, keepdims=True)
    std = np.std(x, axis=-1, keepdims=True)
    return (x - mean) / (std + epsilon)

# Example of residual connection with layer norm
def residual_connection(x, sublayer_output):
    """
    Residual connection: output = LayerNorm(x + Sublayer(x))
    """
    return layer_norm(x + sublayer_output)

# Example
normalized = layer_norm(X)
print(f"Original mean: {np.mean(X, axis=-1)}")
print(f"Original std: {np.std(X, axis=-1)}")
print(f"\nNormalized mean: {np.mean(normalized, axis=-1)}")
print(f"Normalized std: {np.std(normalized, axis=-1)}")

# Residual connection example
residual_output = residual_connection(X, multi_output)
print(f"\nResidual connection output shape: {residual_output.shape}")

## 9. Complete Transformer Block

Putting it all together: a complete transformer block includes:
1. Multi-head self-attention
2. Residual connection + Layer norm
3. Feed-forward network
4. Residual connection + Layer norm

In [ ]:
def transformer_block(x, num_heads, d_model, d_ff):
    """
    A complete transformer block
    
    Args:
        x: Input tensor (seq_len, d_model)
        num_heads: Number of attention heads
        d_model: Model dimension
        d_ff: Feed-forward dimension
    
    Returns:
        output: Transformed output (seq_len, d_model)
    """
    # 1. Multi-head self-attention
    attention_output, _ = multi_head_attention(x, num_heads, d_model)
    
    # 2. Add & Norm
    x = residual_connection(x, attention_output)
    
    # 3. Feed-forward network
    ff_output = feed_forward_network(x, d_model, d_ff)
    
    # 4. Add & Norm
    output = residual_connection(x, ff_output)
    
    return output

# Example: Process input through a transformer block
block_output = transformer_block(X, num_heads=2, d_model=8, d_ff=32)

print(f"Input shape: {X.shape}")
print(f"Transformer block output shape: {block_output.shape}")
print("\nThe transformer block maintains the input shape while transforming the representations.")

## 10. Training Process

LLMs are typically trained using:

### Pretraining (Self-Supervised Learning)
- **Next Token Prediction**: Given a sequence, predict the next token
- **Massive datasets**: Billions to trillions of tokens from the internet
- **Compute intensive**: Thousands of GPUs for weeks/months

### Loss Function
Cross-entropy loss between predicted and actual next token

In [ ]:
def cross_entropy_loss(predictions, targets):
    """
    Compute cross-entropy loss
    
    Args:
        predictions: Model output probabilities (vocab_size,)
        targets: Target token ID (scalar)
    
    Returns:
        loss: Cross-entropy loss
    """
    # Clip predictions to avoid log(0)
    epsilon = 1e-10
    predictions = np.clip(predictions, epsilon, 1 - epsilon)
    
    # Cross-entropy loss
    loss = -np.log(predictions[targets])
    return loss

# Example
vocab_size = len(vocabulary)
predicted_probs = softmax(np.random.randn(vocab_size))
true_token_id = 5

loss = cross_entropy_loss(predicted_probs, true_token_id)
print(f"Predicted probability for true token: {predicted_probs[true_token_id]:.4f}")
print(f"Cross-entropy loss: {loss:.4f}")
print("\nLower loss means better predictions")

## 11. Inference: Text Generation

During inference, LLMs generate text token by token:

1. Start with prompt tokens
2. Model predicts next token probabilities
3. Sample a token from the distribution
4. Add token to sequence
5. Repeat until stopping condition

In [ ]:
def sample_token(logits, temperature=1.0):
    """
    Sample a token from the model's output logits
    
    Args:
        logits: Raw model outputs (vocab_size,)
        temperature: Controls randomness (higher = more random)
    
    Returns:
        Sampled token ID
    """
    # Handle deterministic sampling when temperature is non-positive
    if temperature <= 0:
        return int(np.argmax(logits))

    # Apply temperature
    logits = logits / temperature
    
    # Convert to probabilities
    probs = softmax(logits)
    
    # Sample from distribution
    token_id = np.random.choice(len(probs), p=probs)
    
    return token_id

# Example: Different temperature effects
np.random.seed(42)
logits = np.random.randn(vocab_size)

print("Sampling with different temperatures:")
print("\nLow temperature (0.5) - More deterministic:")
for _ in range(5):
    token_id = sample_token(logits, temperature=0.5)
    print(f"  Token: {id_to_token[token_id]}")

print("\nHigh temperature (1.5) - More random:")
for _ in range(5):
    token_id = sample_token(logits, temperature=1.5)
    print(f"  Token: {id_to_token[token_id]}")

In [ ]:
def top_k_sampling(logits, k=5, temperature=1.0):
    """
    Sample from top-k most likely tokens
    
    Args:
        logits: Model outputs
        k: Number of top tokens to consider
        temperature: Sampling temperature
    """
    # Apply temperature
    logits = logits / temperature
    
    # Get top-k indices
    top_k_indices = np.argsort(logits)[-k:]
    top_k_logits = logits[top_k_indices]
    
    # Convert to probabilities
    top_k_probs = softmax(top_k_logits)
    
    # Sample from top-k
    sampled_index = np.random.choice(len(top_k_probs), p=top_k_probs)
    token_id = top_k_indices[sampled_index]
    
    return token_id

# Example
print("Top-K sampling (k=3):")
for _ in range(5):
    token_id = top_k_sampling(logits, k=3)
    print(f"  Token: {id_to_token[token_id]}")

## 12. Key Concepts Summary

### Core Components:
1. **Tokenization**: Breaking text into processable units
2. **Embeddings**: Converting tokens to dense vectors
3. **Positional Encoding**: Adding position information
4. **Self-Attention**: Learning token relationships
5. **Multi-Head Attention**: Multiple attention perspectives
6. **Feed-Forward Networks**: Non-linear transformations
7. **Layer Normalization**: Stabilizing training
8. **Residual Connections**: Enabling deep networks

### Training:
- Self-supervised on massive text corpora
- Next token prediction objective
- Cross-entropy loss
- Gradient descent optimization

### Inference:
- Autoregressive generation (one token at a time)
- Sampling strategies (temperature, top-k)
- Beam search for better quality

### Scale Matters:
- Larger models learn more patterns
- More data improves performance
- Emergent abilities at scale
- Billions of parameters, trillions of tokens

## 13. Advanced Topics (Brief Overview)

### Fine-tuning
- Adapting pretrained models to specific tasks
- Requires much less data and compute than pretraining
- Instruction tuning, RLHF (Reinforcement Learning from Human Feedback)

### Prompt Engineering
- Crafting effective prompts for better outputs
- Few-shot learning (providing examples)
- Chain-of-thought prompting

### Model Architectures
- **Decoder-only** (GPT): Generate text
- **Encoder-only** (BERT): Understand text
- **Encoder-Decoder** (T5): Transform text

### Optimization Techniques
- KV-cache for faster generation
- Flash Attention for efficiency
- Model quantization (reducing precision)
- Model distillation (creating smaller models)

## 14. Resources for Further Learning

### Papers:
- "Attention Is All You Need" (Vaswani et al., 2017) - Original Transformer paper
- "BERT: Pre-training of Deep Bidirectional Transformers" (Devlin et al., 2018)
- "Language Models are Few-Shot Learners" (Brown et al., 2020) - GPT-3 paper

### Courses:
- Stanford CS224N: Natural Language Processing with Deep Learning
- Fast.ai: Practical Deep Learning for Coders
- DeepLearning.AI: Natural Language Processing Specialization

### Libraries:
- **Hugging Face Transformers**: Easy access to pretrained models
- **PyTorch / TensorFlow**: Deep learning frameworks
- **JAX**: High-performance machine learning

### Implementations:
- nanoGPT by Andrej Karpathy: Minimal GPT implementation
- The Illustrated Transformer by Jay Alammar
- Annotated Transformer

## Conclusion

This notebook covered the fundamental concepts of how LLMs work:

✓ **Tokenization**: Breaking text into processable units  
✓ **Embeddings**: Representing tokens as vectors  
✓ **Positional Encoding**: Adding sequence order information  
✓ **Self-Attention**: The core mechanism for understanding context  
✓ **Transformer Architecture**: How all pieces fit together  
✓ **Training & Inference**: How models learn and generate text  

These concepts form the foundation of modern LLMs. Understanding them provides insight into how AI systems like GPT, Claude, and others work under the hood.

**Next Steps:**
- Experiment with the code examples
- Implement a simple transformer from scratch
- Explore pretrained models with Hugging Face
- Dive deeper into specific topics (attention, training, fine-tuning)